- QA task comes in many flavors, but the one we’ll focus on in this section is called _extractive_ question answering. This involves posing questions about a document and identifying the answers as _spans of text_ in the document itself.
- Encoder-only models like BERT tend to be great at extracting answers to factoid questions like “Who invented the Transformer architecture?” but fare poorly when given open-ended questions like “Why is the sky blue?”
    - In these more challenging cases, encoder-decoder models like T5 and BART are typically used to synthesize the information in a way that’s quite similar to text summarization.
    - TODO: practice this type of _generative_ question answering using [this demo](https://yjernite.github.io/lfqa.html) based on the [ELI5 dataset](https://huggingface.co/datasets/eli5).

# Load dependencies

In [ ]:
# !pip install -q evaluate

In [ ]:
import os, evaluate, torch
import numpy as np
from tqdm.auto import tqdm
from kaggle_secrets import UserSecretsClient
from collections import defaultdict
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer
from transformers import AutoModelForQuestionAnswering
from transformers import TrainingArguments
from transformers import Trainer

# HF login

In [ ]:
secret_label = "HF_TOKEN"
os.environ[secret_label] = UserSecretsClient().get_secret(secret_label)
!hf auth whoami

# Config

In [ ]:
hf_user_name = "amin-oj"
train_sample_size = 8 * 1024
validation_sample_size = 1 * 1024
dataset_dict = dict(path = "squad")
model_checkpoint = "bert-base-cased"
push_to_hub=False
train_bs = 32
eval_bs = 32
num_train_epochs = 3
learning_rate = 2e-5
task = "QA"
ckpt_name = f"{model_checkpoint.split("/")[-1]}-finetuned-{task}-{dataset_dict['path'].split("/")[-1]}"
print(ckpt_name)

# Load data

In [ ]:
train_ds = load_dataset(**dataset_dict, split = f"train[:{train_sample_size}]")
valid_ds = load_dataset(**dataset_dict, split = f"train[:{validation_sample_size}]")
raw_datasets = DatasetDict({
    "train": train_ds,
    "validation": valid_ds
})
print(raw_datasets)
print("===============")
print("Context: ", raw_datasets["train"][0]["context"])
print("Question: ", raw_datasets["train"][0]["question"])
print("Answer: ", raw_datasets["train"][0]["answers"])

- The `context` and `question` fields are very straightforward to use.
- The `answers` field is a bit trickier as it comports a dictionary with two fields that are both lists.
    - This is the format that will be expected by the `squad` metric during evaluation
    - if you are using your own data, you don’t necessarily need to worry about putting the answers in the same format.
- The `text` field is rather obvious, and the `answer_start` field contains the starting character index of each answer in the context.
- During training, there is only one possible answer. We can double-check this by using the `Dataset.filter()` method.

In [ ]:
raw_datasets["train"].filter(lambda x: len(x["answers"]["text"]) != 1)

- For evaluation, however, there are several possible answers for each sample, which may be the same or different.
- We won’t dive into the evaluation script as it will all be wrapped up by a 🤗 Datasets metric for us, but the short version is that some of the questions have several possible answers, and this script will compare a predicted answer to all the acceptable answers and take the best score.

In [ ]:
print(raw_datasets["validation"][0]["answers"])
print(raw_datasets["validation"][2]["answers"])

# Prep train data

We’ll be fine-tuning a BERT model, but you can use any other model type as long as it has a fast tokenizer implemented.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
print(tokenizer.is_fast)

In [ ]:
context = raw_datasets["train"][0]["context"]
question = raw_datasets["train"][0]["question"]

inputs = tokenizer(question, context)
tokenizer.decode(inputs["input_ids"])

- The labels will then be the index of the tokens starting and ending the answer
- The model will be tasked to predict one start and end logit per token in the input
- There are examples in the dataset with very long contexts that will exceed the maximum length we set (which is 384 in this case).
- We will deal with long contexts by creating several training features from one sample of our dataset, with a sliding window between them.

- To see how this works using the current example, we can limit the length to 100 and use a sliding window of 50 tokens.
    -   `max_length` to set the maximum length (here 100)
    -   `truncation="only_second"` to truncate the context (which is in the second position) when the question with its context is too long
    -   `stride` to set the number of overlapping tokens between two successive chunks (here 50)
    -   `return_overflowing_tokens=True` to let the tokenizer know we want the overflowing tokens
- The dataset provides us with the start character of the answer in the context, and by adding the length of the answer, we can find the end character in the context.
    - To map those to token indices, we will need to use the offset mappings. We can have our tokenizer return these by passing along `return_offsets_mapping=True`

In [ ]:
inputs = tokenizer(
    question,
    context,
    max_length=100,
    truncation="only_second",
    stride=50,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
)

for ids in inputs["input_ids"]:
    print(tokenizer.decode(ids))

print("===============")
print(list(inputs.keys()))

- Note that the answer to the question (“Bernadette Soubirous”) only appears in the third and last inputs, so by dealing with long contexts in this way, we will create some training examples where the answer is not included in the context.
- For those examples, the labels will be `start_position = end_position = 0` (so we predict the `[CLS]` token).
- We will also set those labels in the unfortunate case where the answer has been truncated, so that we only have the start (or end) of it.
- For the examples where the answer is fully in the context, the labels will be the index of the token where the answer starts and the index of the token where the answer ends.

In [ ]:
# here we only tokenized one example
print(inputs["overflow_to_sample_mapping"])

In [ ]:
# But if we tokenize more examples, this will become more useful
inputs = tokenizer(
    raw_datasets["train"][2:6]["question"],
    raw_datasets["train"][2:6]["context"],
    max_length=100,
    truncation="only_second",
    stride=50,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
)
print(f"The 4 examples gave {len(inputs['input_ids'])} features.")
print(f"Here is where each comes from: {inputs['overflow_to_sample_mapping']}.")

The `overflow_to_sample_mapping` will be useful to map each feature we get to its corresponding label. As mentioned earlier, those labels are:

-   `(0, 0)` if the answer is not in the corresponding span of the context
-   `(start_position, end_position)` if the answer is in the corresponding span of the context, with `start_position` being the index of the token (in the input IDs) at the start of the answer and `end_position` being the index of the token (in the input IDs) where the answer ends

To determine which of these is the case and, if relevant, the positions of the tokens, we first find the indices that start and end the context in the input IDs. We could use the token type IDs to do this, but since those do not necessarily exist for all models (DistilBERT does not require them, for instance), we’ll instead use the `sequence_ids()` method of the `BatchEncoding` our tokenizer returns.

In [ ]:
answers = raw_datasets["train"][2:6]["answers"]

print(len(inputs["offset_mapping"]))
print(len(inputs["overflow_to_sample_mapping"]))
print(inputs["overflow_to_sample_mapping"])
print(len(answers))
i = 4
print("==============")
print(inputs["offset_mapping"][i])
print("==============")
print(inputs["overflow_to_sample_mapping"][i])
print("==============")
print(answers[inputs["overflow_to_sample_mapping"][i]])
print("==============")
print(inputs.sequence_ids(i))

In [ ]:
start_positions = []
end_positions = []
for i, offset in enumerate(inputs["offset_mapping"]):
    sample_idx = inputs["overflow_to_sample_mapping"][i]
    answer = answers[sample_idx]
    start_char = answer["answer_start"][0]
    end_char = answer["answer_start"][0] + len(answer["text"][0])
    sequence_ids = inputs.sequence_ids(i)

    # Find the start and end of the context
    idx = 0
    while sequence_ids[idx] != 1:
        idx += 1
    context_start = idx
    while sequence_ids[idx] == 1:
        idx += 1
    context_end = idx - 1

    # If the answer is not fully inside the context, label is (0, 0)
    if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
        start_position = 0
        end_position = 0
    else:
        # Otherwise it's the start and end token positions
        idx = context_start
        while idx <= context_end and offset[idx][0] <= start_char:
            idx += 1
        start_position = idx - 1

        idx = context_end
        while idx >= context_start and offset[idx][1] >= end_char:
            idx -= 1
        end_position = idx + 1

    start_positions.append(start_position)
    end_positions.append(end_position)

print(start_positions)
print(end_positions)

In [ ]:
def find_answer_positions(sequence_ids, offset, start_char, end_char):
    if 1 not in sequence_ids:
        return np.int64(0), np.int64(0)
    context_start = sequence_ids.index(1)
    context_end = sequence_ids.index(None, context_start) -1

    offset = np.array(offset)

    # print(f"context_start: {context_start} | context_end: {context_end}")
    
    # If the answer is not fully inside the context, label it (0, 0)
    if (offset[context_start][0] > end_char) or (offset[context_end][1] < start_char):
        return np.int64(0), np.int64(0)
    else:
        start_position = context_start + np.argmax(offset[context_start:context_end+1, 0] == start_char)
        end_position = context_start + np.argmax(offset[context_start:context_end+1, 1] == end_char)

    return start_position, end_position


start_positions = []
end_positions = []
for i, offset in enumerate(inputs["offset_mapping"]):
    sample_idx = inputs["overflow_to_sample_mapping"][i]
    answer = answers[sample_idx]
    start_char = answer["answer_start"][0]
    end_char = answer["answer_start"][0] + len(answer["text"][0])
    sequence_ids = inputs.sequence_ids(i)

    start_position, end_position = find_answer_positions(
        sequence_ids, offset, start_char, end_char
    )

    start_positions.append(start_position)
    end_positions.append(end_position)

print(start_positions)
print(end_positions)

Let’s take a look at a few results to verify that our approach is correct. For the first feature we find `(83, 85)` as labels, so let’s compare the theoretical answer with the decoded span of tokens from 83 to 85 (inclusive):

In [ ]:
idx = 0
sample_idx = inputs["overflow_to_sample_mapping"][idx]
answer = answers[sample_idx]["text"][0]

start = start_positions[idx]
end = end_positions[idx]
labeled_answer = tokenizer.decode(inputs["input_ids"][idx][start : end + 1])


print(f"Theoretical answer: {answer}, labels give: {labeled_answer}")

So that’s a match! Now let’s check index 4, where we set the labels to `(0, 0)`, which means the answer is not in the context chunk of that feature:

In [ ]:
idx = 4
sample_idx = inputs["overflow_to_sample_mapping"][idx]
answer = answers[sample_idx]["text"][0]
start = start_positions[idx]
end = end_positions[idx]
decoded_example = tokenizer.decode(inputs["input_ids"][idx])
labeled_answer = tokenizer.decode(inputs["input_ids"][idx][start : end + 1])
print(f"Theoretical answer: {answer}, labels give: {labeled_answer}\ndecoded example: {decoded_example}")

## Preprocess function

- Now that we have seen step by step how to preprocess our training data, we can group it in a function we will apply on the whole training dataset.
- We’ll pad every feature to the maximum length we set
    - as most of the contexts will be long (and the corresponding samples will be split into several features)
    - So, there is no real benefit to applying dynamic padding here.

In [ ]:
def preprocess_training_examples(
    examples,
    tokenizer,
    max_length = 384,
    stride = 128
):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = answers[sample_idx]
        start_char = answer["answer_start"][0]
        end_char = answer["answer_start"][0] + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)
    
        start_position, end_position = find_answer_positions(
            sequence_ids, offset, start_char, end_char
        )
    
        start_positions.append(start_position)
        end_positions.append(end_position)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    
    return inputs

In [ ]:
train_dataset = raw_datasets["train"].map(
    preprocess_training_examples,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
    fn_kwargs = {"tokenizer": tokenizer},
)

In [ ]:
print(len(raw_datasets["train"]))
print(len(train_dataset))

# Evaluation

## Prep the validation data

- Preprocessing the validation data will be slightly easier
    - as we don’t need to generate labels (unless we want to compute a validation loss, but that number won’t really help us understand how good the model is).
    - The real joy will be to interpret the predictions of the model into spans of the original context.
    - For this, we will just need to
        - store both the offset mappings
        - and some way to match each created feature to the original example it comes from.
            - Since there is an ID column in the original dataset, we’ll use that ID.

- The only thing we’ll add here is a tiny bit of cleanup of the offset mappings.
    - Once we’re in the post-processing stage (shown and discussed below), we won’t have any way to know which part of the input IDs corresponded to the context and which part was the question
    - Hence, we use the `sequence_ids()` method available for the output of the tokenizer to set the offsets corresponding to the question to `None`.

In [ ]:
def preprocess_validation_examples(
    examples,
    tokenizer,
    max_length = 384,
    stride = 128
):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = inputs.pop("overflow_to_sample_mapping")
    example_ids = []

    for i in range(len(inputs["input_ids"])):
        sample_idx = sample_map[i]
        example_ids.append(examples["id"][sample_idx])

        sequence_ids = inputs.sequence_ids(i)
        offset = inputs["offset_mapping"][i]
        inputs["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None for k, o in enumerate(offset)
        ]

    inputs["example_id"] = example_ids
    return inputs

In [ ]:
validation_dataset = raw_datasets["validation"].map(
    preprocess_validation_examples,
    batched=True,
    remove_columns=raw_datasets["validation"].column_names,
    fn_kwargs = {"tokenizer": tokenizer},
)

In [ ]:
print(len(raw_datasets["validation"]))
print(len(validation_dataset))

## Post-processing

The model will output logits for the start and end positions of the answer in the input IDs, as we saw during our exploration of the [`question-answering` pipeline](https://github.com/aminojagh/HFLLM/blob/main/notebooks/tokenization_deepdive/2_fast_tokenizers_powers.ipynb). The post-processing step will be similar to what we did there, so here’s a quick reminder of the actions we took:

-   We masked the start and end logits corresponding to tokens outside of the context.
-   We then converted the start and end logits into probabilities using a softmax.
-   We attributed a score to each `(start_token, end_token)` pair by taking the product of the corresponding two probabilities.
-   We looked for the pair with the maximum score that yielded a valid answer (e.g., a `start_token` lower than `end_token`).

Here we will change this process slightly because we don’t need to compute actual scores (just the predicted answer).
- This means we can skip the softmax step.
- To go faster, we also won’t score all the possible `(start_token, end_token)` pairs, but only the ones corresponding to the highest `n_best` logits (with `n_best=20`).
- Since we will skip the softmax, those scores will be logit scores, and will be obtained by taking the sum of the start and end logits instead of the product.
---
To demonstrate all of this, we will need some kind of predictions. Since we have not trained our model yet, we are going to use the default model for the QA pipeline to generate some predictions on a small part of the validation set.

In [ ]:
small_eval_set = raw_datasets["validation"].select(range(100))
trained_checkpoint = "distilbert-base-cased-distilled-squad"
temp_tokenizer = AutoTokenizer.from_pretrained(trained_checkpoint)
tokenized_small_eval_set = small_eval_set.map(
    preprocess_validation_examples,
    batched=True,
    remove_columns=raw_datasets["validation"].column_names,
    fn_kwargs = {"tokenizer": temp_tokenizer},
)
tokenized_small_eval_set_for_model = tokenized_small_eval_set\
    .remove_columns(["example_id", "offset_mapping"])
tokenized_small_eval_set_for_model.set_format("torch")

device = torch.device("cuda")
batch = tokenized_small_eval_set_for_model[:len(tokenized_small_eval_set_for_model)]
batch = {k:v.to(device) for k,v in batch.items()}
trained_model = AutoModelForQuestionAnswering\
    .from_pretrained(trained_checkpoint)\
    .to(device)

with torch.no_grad():
    outputs = trained_model(**batch)

- Since the `Trainer` will give us predictions as NumPy arrays, we grab the start and end logits and convert them to that format.
- We also need to find the predicted answer for each example in our `small_eval_set`. One example may have been split into several features in `tokenized_small_eval_set`, so the first step is to map each example in `small_eval_set` to the corresponding features in `tokenized_small_eval_set`.

In [ ]:
start_logits = outputs.start_logits.cpu().numpy()
end_logits = outputs.end_logits.cpu().numpy()

example_to_features = defaultdict(list)
for idx, feature in enumerate(tokenized_small_eval_set):
    example_to_features[feature["example_id"]].append(idx)

With this in hand, we can really get to work by looping through all the examples and, for each example, through all the associated features. As we said before, we’ll look at the logit scores for the `n_best` start logits and end logits, excluding positions that give:

-   An answer that wouldn’t be inside the context
-   An answer with negative length
-   An answer that is too long (we limit the possibilities at `max_answer_length=30`)

Once we have all the scored possible answers for one example, we just pick the one with the best logit score.

In [ ]:
n_best = 20
max_answer_length = 30
predicted_answers = []

for example in small_eval_set:
    example_id = example["id"]
    context = example["context"]
    answers = []

    for feature_index in example_to_features[example_id]:
        start_logit = start_logits[feature_index]
        end_logit = end_logits[feature_index]
        offsets = tokenized_small_eval_set["offset_mapping"][feature_index]

        start_indexes = np.argsort(start_logit)[-1 : -n_best - 1 : -1].tolist()
        end_indexes = np.argsort(end_logit)[-1 : -n_best - 1 : -1].tolist()
        for start_index in start_indexes:
            for end_index in end_indexes:
                # Skip answers that are not fully in the context
                if (offsets[start_index] is None) or (offsets[end_index] is None):
                    continue
                # Skip answers with a length that is either < 0 or > max_answer_length.
                if (
                    (end_index < start_index)
                    or (end_index - start_index + 1 > max_answer_length)
                ):
                    continue

                answers.append(
                    {
                        "text": context[offsets[start_index][0] : offsets[end_index][1]],
                        "logit_score": start_logit[start_index] + end_logit[end_index],
                    }
                )

    best_answer = max(answers, key=lambda x: x["logit_score"])
    predicted_answers.append({"id": example_id, "prediction_text": best_answer["text"]})

## Compute metric

In [ ]:
metric = evaluate.load("squad")

theoretical_answers = [
    {"id": ex["id"], "answers": ex["answers"]}
    for ex in small_eval_set
]

print(predicted_answers[0])
print(theoretical_answers[0])

print("==================")
print(metric.compute(predictions=predicted_answers, references=theoretical_answers))

- Now let’s put everything we just did in a `compute_metrics()` function that we will use in the `Trainer`.
- Normally, that `compute_metrics()` function only receives a tuple `eval_preds` with logits and labels.
- Here we will need a bit more, as we have to look in the dataset of features for the offset and in the dataset of examples for the original contexts
- So, we won’t be able to use this function to get regular evaluation results during training.
    - We will only use it at the end of training to check the results.
    - TODO: Write our own subclass of `Trainer` to do this (an approach you can find in the [question answering example script](https://github.com/huggingface/transformers/blob/master/examples/pytorch/question-answering/trainer_qa.py))
- The `compute_metrics()` function groups the same steps as before
    - we just add a small check in case we don’t come up with any valid answers
        - in which case we predict an empty string.

In [ ]:
def compute_metrics(
    start_logits, end_logits,
    tokenized_eval_examples, eval_examples
):
    example_to_features = defaultdict(list)
    for idx, feature in enumerate(tokenized_eval_examples):
        example_to_features[feature["example_id"]].append(idx)

    predicted_answers = []
    for example in tqdm(eval_examples):
        example_id = example["id"]
        context = example["context"]
        answers = []

        # Loop through all tokenized_eval_examples associated with that example
        for feature_index in example_to_features[example_id]:
            start_logit = start_logits[feature_index]
            end_logit = end_logits[feature_index]
            offsets = tokenized_eval_examples[feature_index]["offset_mapping"]

            start_indexes = np.argsort(start_logit)[-1 : -n_best - 1 : -1].tolist()
            end_indexes = np.argsort(end_logit)[-1 : -n_best - 1 : -1].tolist()
            for start_index in start_indexes:
                for end_index in end_indexes:
                    # Skip answers that are not fully in the context
                    if offsets[start_index] is None or offsets[end_index] is None:
                        continue
                    # Skip answers with a length that is either < 0 or > max_answer_length
                    if (
                        end_index < start_index
                        or end_index - start_index + 1 > max_answer_length
                    ):
                        continue

                    answer = {
                        "text": context[offsets[start_index][0] : offsets[end_index][1]],
                        "logit_score": start_logit[start_index] + end_logit[end_index],
                    }
                    answers.append(answer)

        # Select the answer with the best score
        if len(answers) > 0:
            best_answer = max(answers, key=lambda x: x["logit_score"])
            predicted_answers.append(
                {"id": example_id, "prediction_text": best_answer["text"]}
            )
        else:
            predicted_answers.append({"id": example_id, "prediction_text": ""})

    theoretical_answers = [{"id": ex["id"], "answers": ex["answers"]} for ex in eval_examples]
    return metric.compute(predictions=predicted_answers, references=theoretical_answers)

In [ ]:
compute_metrics(start_logits, end_logits, tokenized_small_eval_set, small_eval_set)

# Fine-tuning the model with the Trainer API

- Since we padded all the samples to the maximum length we set, there is no data collator to define.

In [ ]:
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

- As usual, we get a warning that some weights are not used (the ones from the pretraining head) and some others are initialized randomly (the ones for the question answering head).
- We should be used to this by now, but that means this model is not ready to be used just yet and needs fine-tuning

In [ ]:
args = TrainingArguments(
    ckpt_name,
    learning_rate=learning_rate,
    num_train_epochs=num_train_epochs,
    push_to_hub=push_to_hub,
    eval_strategy="no",
    save_strategy="epoch",
    weight_decay=0.01,
    bf16=True,
)


trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
)

In [ ]:
trainer.train()

## Evaluation

In [ ]:
predictions, _, _ = trainer.predict(validation_dataset)
start_logits, end_logits = predictions
compute_metrics(start_logits, end_logits, validation_dataset, raw_datasets["validation"])

## Push the model to the hub

In [ ]:
if push_to_hub: trainer.push_to_hub(commit_message="Training complete")

# Using the fine-tuned model

In [ ]:
from transformers import pipeline

# Replace this with your own checkpoint
model_checkpoint = "huggingface-course/bert-finetuned-squad"
question_answerer = pipeline("question-answering", model=model_checkpoint)

context = """
🤗 Transformers is backed by the three most popular deep learning libraries — Jax, PyTorch and TensorFlow — with a seamless integration
between them. It's straightforward to train your models with one before loading them for inference with the other.
"""
question = "Which deep learning libraries back 🤗 Transformers?"
question_answerer(question=question, context=context)